<a href="https://colab.research.google.com/github/Prachichoudhary28/Data-Analytics-Portfolio/blob/main/customer_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
df= pd.read_csv('customer_shopping_behavior.csv')


In [ ]:
df['Review Rating']= df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [ ]:
df.columns=df.columns.str.lower()
df.columns=df.columns.str.replace(' ','_')
df=df.rename(columns={'purchase_amount_(usd)':'purchase_amount'})

In [ ]:
labels =["Young Adult" ,"Adult" ,"Middle-Aged","Senior"]
df['age_group']=pd.qcut(df['age'], q=4 , labels=labels)


In [ ]:
frequency_mapping={
    'Fortnightly':14,
    'Weekly':7,
    'Annually':365,
    'Quarterly':90,
    'Bi-Weekly':60,
    'Monthly':30,
    'Every 3 Months':90
}
df['purchase_frequency']=df['frequency_of_purchases'].map(frequency_mapping)

In [ ]:
df.head()

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,promo_code_used,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly,Middle-Aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly,Young Adult,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly,Middle-Aged,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly,Young Adult,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually,Middle-Aged,365


In [ ]:
import sqlite3

connection = sqlite3.connect('customer_shopping.db')

In [ ]:
df.to_sql(
    'customer_shopping',
    connection,
    if_exists='replace',
    index=False
)

3900

In [ ]:
query = """
SELECT *
FROM customer_shopping
LIMIT 10;
"""

result = pd.read_sql(query, connection)

result

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,promo_code_used,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly,Middle-Aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly,Young Adult,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly,Middle-Aged,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly,Young Adult,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually,Middle-Aged,365
5,6,46,Male,Sneakers,Footwear,20,Wyoming,M,White,Summer,2.9,Yes,Standard,Yes,Yes,14,Venmo,Weekly,Middle-Aged,7
6,7,63,Male,Shirt,Clothing,85,Montana,M,Gray,Fall,3.2,Yes,Free Shipping,Yes,Yes,49,Cash,Quarterly,Senior,90
7,8,27,Male,Shorts,Clothing,34,Louisiana,L,Charcoal,Winter,3.2,Yes,Free Shipping,Yes,Yes,19,Credit Card,Weekly,Young Adult,7
8,9,26,Male,Coat,Outerwear,97,West Virginia,L,Silver,Summer,2.6,Yes,Express,Yes,Yes,8,Venmo,Annually,Young Adult,365
9,10,57,Male,Handbag,Accessories,31,Missouri,M,Pink,Spring,4.8,Yes,2-Day Shipping,Yes,Yes,4,Cash,Quarterly,Middle-Aged,90


In [ ]:
# Question 1 : What is total revenue generated by male vs female customers ?

query = """
SELECT
    gender,
    SUM(purchase_amount) AS total_revenue
FROM customer_shopping
GROUP BY gender;
"""

result = pd.read_sql(query, connection)

result

,gender,total_revenue
0,Female,75191
1,Male,157890


In [ ]:
# Question 2 : Which customers uses the discount but still spent more than average purchase amount?

query = """
SELECT
    customer_id,
    purchase_amount
FROM customer_shopping
WHERE purchase_amount > (
    SELECT AVG(purchase_amount)
    FROM customer_shopping
);
"""

result = pd.read_sql(query, connection)

result

,customer_id,purchase_amount
0,2,64
1,3,73
2,4,90
3,7,85
4,9,97
...,...,...
1958,3893,86
1959,3894,64
1960,3895,78
1961,3899,77


In [ ]:
# Question 3 : Which are the top 5 products with highest average review ratings?

query = """
SELECT
    item_purchased,
    AVG(review_rating) AS average_rating
FROM customer_shopping
GROUP BY item_purchased
ORDER BY average_rating DESC
LIMIT 5;
"""

result = pd.read_sql(query, connection)

result

,item_purchased,average_rating
0,Gloves,3.861429
1,Sandals,3.844375
2,Boots,3.818750
3,Hat,3.801299
4,Skirt,3.784810


In [ ]:
# Question 4 : Compare average purchases amount between Standard and Express shipping?

query = """
SELECT
    shipping_type,
    AVG(purchase_amount) AS average_purchase_amount
FROM customer_shopping
WHERE shipping_type IN ('Express', 'Standard')
GROUP BY shipping_type;
"""

result = pd.read_sql(query, connection)

result

,shipping_type,average_purchase_amount
0,Express,60.475232
1,Standard,58.460245


In [ ]:
 # Question 5 : Do susbscribed customers spend more? Compare average spent and total revenue between subscribers and non-subscribers .

query = """
SELECT
    subscription_status,
    SUM(purchase_amount) AS total_revenue,
    AVG(purchase_amount) AS average_spent
FROM customer_shopping
GROUP BY subscription_status;
"""

result = pd.read_sql(query, connection)

result

,subscription_status,total_revenue,average_spent
0,No,170436,59.865121
1,Yes,62645,59.491928


In [ ]:
# Question 6 : Which top 5 products have highest percentage of purchase with discount applied ?

query = """
SELECT
    item_purchased,
    SUM(CASE WHEN discount_applied = 'Yes' THEN 1 ELSE 0 END) * 100.0
    / COUNT(*) AS discount_percentage
FROM customer_shopping
GROUP BY item_purchased
ORDER BY discount_percentage DESC
LIMIT 5;
"""

result = pd.read_sql(query, connection)

result

,item_purchased,discount_percentage
0,Hat,50.000000
1,Sneakers,49.655172
2,Coat,49.068323
3,Sweater,48.170732
4,Pants,47.368421


In [ ]:
# Question 7 : Segement customers as New , Returning and Loyal based on their total number of previous purchases , and show count of each segment .

query = """
SELECT
    CASE
        WHEN previous_purchases =1 THEN 'NEW'
        WHEN previous_purchases BETWEEN 2 AND 5 THEN 'RETURNING'
        ELSE 'LOYAL'
    END AS customer_segment,
    COUNT(DISTINCT customer_id) AS customer_count
FROM customer_shopping
GROUP BY customer_segment;
"""

result = pd.read_sql(query, connection)

result

,customer_segment,customer_count
0,LOYAL,3476
1,NEW,83
2,RETURNING,341


In [ ]:
# Question 8 : What are  the top 3 most purchased product in each category ?

query = """
WITH product_counts AS (
    SELECT
        category,
        item_purchased,
        COUNT(*) AS purchase_count
    FROM customer_shopping
    GROUP BY category, item_purchased
),

ranked_products AS (
    SELECT
        category,
        item_purchased,
        purchase_count,
        ROW_NUMBER() OVER (
            PARTITION BY category
            ORDER BY purchase_count DESC
        ) AS product_rank
    FROM product_counts
)

SELECT
    category,
    item_purchased,
    purchase_count
FROM ranked_products
WHERE product_rank <= 3
ORDER BY category, product_rank;
"""

result = pd.read_sql(query, connection)

result

,category,item_purchased,purchase_count
0,Accessories,Jewelry,171
1,Accessories,Belt,161
2,Accessories,Sunglasses,161
3,Clothing,Blouse,171
4,Clothing,Pants,171
5,Clothing,Shirt,169
6,Footwear,Sandals,160
7,Footwear,Shoes,150
8,Footwear,Sneakers,145
9,Outerwear,Jacket,163


In [ ]:
# Question 9 : Are customer who are Repeat Buyer (previous purchase > 5) also likely to subscribe?

query = """
SELECT
    subscription_status,
    COUNT(DISTINCT customer_id) AS repeat_customer_count
FROM customer_shopping
WHERE previous_purchases > 5
GROUP BY subscription_status;
"""

result = pd.read_sql(query, connection)

result

,subscription_status,repeat_customer_count
0,No,2518
1,Yes,958


In [ ]:
# Question 10 : What is revenue contribution of each age group?

query = """
SELECT
    age_group,
    SUM(purchase_amount) AS total_revenue
FROM customer_shopping
GROUP BY age_group
ORDER BY total_revenue DESC;
"""

result = pd.read_sql(query, connection)

result

,age_group,total_revenue
0,Young Adult,62143
1,Middle-Aged,59197
2,Adult,55978
3,Senior,55763


In [ ]:
df.to_csv('customer_shopping_cleaned.csv', index=False)

In [ ]:
from google.colab import files

files.download('customer_shopping_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>